# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import pandas as pd
import numpy as np
import faiss
from google import genai
from google.genai import types
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env')

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [3]:
def get_embedding(text):
    result = client.models.embed_content(
        model='gemini-embedding-001',
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [4]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

faiss.write_index(index, '../data/ecommerce_faq.index')
print('Index tersimpan di ../data/ecommerce_faq.index')

print('Jumlah vektor tersimpan di index:', index.ntotal)

Index tersimpan di ../data/ecommerce_faq.index
Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [5]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.33353505)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.40880978)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.4320181)}


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [6]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ {r['rank']}: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Persona: Kamu adalah asisten customer service e-commerce yang ramah dan profesional.

Task: Jawab pertanyaan pengguna HANYA berdasarkan Context di bawah. Jika informasi tidak ditemukan dalam Context, katakan dengan jujur bahwa kamu belum memiliki informasi tersebut dan sarankan pengguna menghubungi customer service — jangan mengarang jawaban dari pengetahuan umum.

Context:
{context}

Format: Jawab dalam bahasa Indonesia yang natural dan ringkas (2-4 kalimat atau langkah singkat), langsung menjawab pertanyaan. Jangan menambahkan informasi di luar Context.

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

In [12]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Jawaban LLM:
Halo! Ya, fitur cicilan tersedia untuk transaksi menggunakan kartu kredit tertentu dengan nominal minimal yang ditentukan oleh pihak bank. Silakan pastikan kartu kredit Anda memenuhi ketentuan tersebut saat melakukan pembayaran.

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [14]:
test_queries = [
    # 1-2 query langsung (frasa mirip FAQ)
    "Apa saja metode pembayaran yang tersedia?",
    "Berapa lama estimasi waktu pengiriman?",
    # 1-2 query parafrase/sinonim (kata-kata BEDA dari FAQ)
    "gimana cara tahu pesananku sudah dikirim ke mana?",      # → FAQ lacak resi/pengiriman
    "kalau barang yang datang cacat atau beda gambar gimana?", # → FAQ pengembalian
    # 1 query di LUAR knowledge base
    "Apakah bisa tukar tambah HP lama saya dengan produk baru?",
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Hasil retrieval:')
    for s in sources:
        print(f" {s['rank']}. {s['question']} (distance: {s['distance']:.4f})")
    print('Jawaban:', answer)
    print('---')

Query: Apa saja metode pembayaran yang tersedia?
Hasil retrieval:
 1. Metode pembayaran apa saja yang tersedia? (distance: 0.3127)
 2. Apakah bisa membayar menggunakan lebih dari satu metode sekaligus? (distance: 0.5027)
 3. Bagaimana cara melakukan pemesanan di platform ini? (distance: 0.5846)
Jawaban: Halo! Saat ini kami mendukung metode pembayaran melalui transfer bank, kartu kredit/debit, e-wallet, serta pembayaran di minimarket rekanan. Silakan pilih metode yang paling nyaman bagi Anda saat melakukan checkout.
---
Query: Berapa lama estimasi waktu pengiriman?
Hasil retrieval:
 1. Berapa lama estimasi waktu pengiriman barang? (distance: 0.2160)
 2. Bagaimana jika paket saya belum sampai melebihi estimasi? (distance: 0.4327)
 3. Berapa lama waktu yang dibutuhkan sebelum pesanan diproses penjual? (distance: 0.5084)
Jawaban: Estimasi pengiriman umumnya membutuhkan waktu 2-5 hari kerja untuk area dalam kota dan 5-10 hari kerja untuk pengiriman luar pulau, tergantung pada jasa kirim yan

### Catatan Analisis Relevansi

Catatan kecil: angka distance di bawah ini tingkat kemiripan dari FAISS, makin kecil berarti pertanyaan dan FAQ-nya makin mirip.

**1. Apa saja metode pembayaran yang tersedia?** (ada jawabannya di FAQ)

FAQ yang ketemu langsung yang dimaksud (distance 0.31), dan jawabannya persis isi FAQ: transfer bank, kartu kredit/debit, e-wallet, dan minimarket. Aman.

**2. Berapa lama estimasi waktu pengiriman?** (ada jawabannya di FAQ)

Sama, langsung kena (0.21, paling mirip dari semua percobaan). Jawabannya benar: 2-5 hari dalam kota, 5-10 hari luar pulau.

**3. gimana cara tahu pesananku sudah dikirim ke mana?** (pakai bahasa sendiri, beda dari FAQ)

Di pertanyaan ini tidak ada kata lacak atau resi sama sekali, tapi FAQ yang ketemu tetap soal melacak paket (0.38). Jadi sistemnya mencari maknanya, bukan kata yang kebetulan sama. Jawabannya juga benar soal lacak paket lewat resi di halaman Pesanan Saya.

**4. kalau barang yang datang cacat atau beda gambar gimana?** (pakai bahasa sendiri, soalnya dua hal sekaligus)

Tiga FAQ yang muncul kebetulan pas semua: barang salah, prosedur pengembalian, dan paket rusak. Tapi jawaban LLM-nya agak keteteran, dua kasus itu dijadikan satu dan yang disebut cuma aturan maksimal 3 hari. Padahal untuk barang rusak di FAQ ada aturan sendiri, maksimal 2x24 jam. Lumayan membantu, tapi tidak lengkap.

**5. Apakah bisa tukar tambah HP lama saya dengan produk baru?** (sengaja yang jawabannya tidak ada di FAQ)

Sistem tetap menampilkan 3 FAQ terdekat soal tukar ukuran/warna, ubah nomor telepon, dan bandingkan produk. Sebenarnya tidak ada yang nyambung, tapi LLM-nya jujur bilang belum punya informasi dan menyuruh hubungi customer service, tidak mengarang. Ini karena di prompt ada aturan hanya boleh jawab dari FAQ.

**Keterbatasan yang saya sadari setelah uji ini:**

- Sistem selalu mengambil 3 FAQ terdekat walaupun sebenarnya tidak ada yang cocok. Belum ada ambang skor untuk langsung bilang tidak tahu, jadi penolakan di percobaan 5 itu masih tergantung LLM-nya patuh atau tidak, belum dijamin dari sistemnya.
- Pertanyaan yang isinya dua hal seperti percobaan 4 hasilnya bisa berubah-ubah. Kali ini ketiga FAQ-nya kebetulan pas semua, tapi kalau cuma ambil satu FAQ teratas pasti jawabannya lebih meleset.
- Kadang FAQ yang tidak nyambung ikut keambil, misalnya di percobaan 1 urutan tiga soal cara memesan. Tidak mengganggu hasil sih, tapi pencariannya memang tidak selalu bersih.